In [ ]:
!pip install langgraph langchain langchain-core langchain-ollama

In [ ]:
# LangGraph Multi-Agent Exercise Recommendation System
# ============================================================
# 질문:
# "체력이 안좋고, 살이 계속 찌는데 어떤 운동을 할까?"
#
# 에이전트 구조:
# 1. 증상 추출 에이전트
# 2. 운동 후보 에이전트
# 3. 답변 생성 에이전트
#
# 추가 기능:
# - grandalf 기반 그래프 구조 시각화


from typing import TypedDict, List

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from IPython.display import Image, display


# 1. LLM 설정

llm = ChatOllama(
    model="exaone3.5:2.4b",
    temperature=0.2
)

parser = StrOutputParser()


# 2. State 정의

class ExerciseState(TypedDict):
    user_question: str
    symptoms: List[str]
    exercise_candidates: List[str]
    final_answer: str



# 3. 증상 추출 에이전트

def symptom_extractor_agent(state: ExerciseState) -> dict:
    """
    사용자의 질문에서 현재 상태/문제만 추출하는 에이전트
    """

    question = state["user_question"]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
            너는 사용자의 질문에서 '현재 겪고 있는 문제/증상'만 추출하는 에이전트다.

            반드시 아래 조건을 지켜라.

            1. 사용자가 현재 겪는 상태만 추출한다.
            2. 해결 방법, 추천 운동, 목표는 추출하지 않는다.
            3. '운동 추천', '체중 감소', '체력 향상' 같은 해결 방향은 제외한다.
            4. 결과는 콤마(,)로 구분된 짧은 키워드로만 출력한다.
            5. 의학적 진단명은 함부로 만들지 않는다.
            6. 설명 문장은 쓰지 않는다.

            예시 1:
            입력: 체력이 안좋고 살이 계속 찌는데 어떤 운동을 할까?
            출력: 체력 저하, 체중 증가

            예시 2:
            입력: 요즘 숨이 빨리 차고 운동을 거의 못 하겠어
            출력: 쉽게 숨참, 운동 부족

            예시 3:
            입력: 허리가 아프고 오래 걷기 힘들어
            출력: 허리 통증, 보행 부담
            """
        ),
        ("human", "{question}")
    ])

    chain = prompt | llm | parser
    result = chain.invoke({"question": question})

    symptoms = [
        item.strip()
        for item in result.replace("\n", ",").split(",")
        if item.strip()
    ]

    print("\n==============================")
    print("[1] 증상 추출 에이전트")
    print("==============================")
    print("추출 결과:", symptoms)

    return {
        "symptoms": symptoms
    }


# 4. 운동 후보 에이전트

def exercise_candidate_agent(state: ExerciseState) -> dict:
    """
    추출된 증상과 목표를 바탕으로 운동 후보 리스트를 생성하는 에이전트
    """

    symptoms = state["symptoms"]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
            너는 사용자의 상태에 맞는 운동 후보를 추천하는 에이전트다.

            사용자의 증상과 목표를 보고 초보자가 할 수 있는 운동 후보를 추천하라.

            조건:
            1. 체력이 낮은 사람도 시작 가능한 운동을 우선 추천한다.
            2. 체중 증가 관리에 도움이 되는 운동을 포함한다.
            3. 과격한 고강도 운동은 제외한다.
            4. 운동명만 콤마(,)로 구분해서 출력한다.
            5. 설명 문장은 쓰지 않는다.

            예시 출력:
            걷기, 실내 자전거, 가벼운 근력운동, 스트레칭
            """
        ),
        ("human", "사용자 증상 및 목표: {symptoms}")
    ])

    chain = prompt | llm | parser
    result = chain.invoke({"symptoms": ", ".join(symptoms)})

    exercise_candidates = [
        item.strip()
        for item in result.replace("\n", ",").split(",")
        if item.strip()
    ]

    print("\n==============================")
    print("[2] 운동 후보 에이전트")
    print("==============================")
    print("추천 후보:", exercise_candidates)

    return {
        "exercise_candidates": exercise_candidates
    }



# 5. 답변 생성 에이전트

def answer_generator_agent(state: ExerciseState) -> dict:
    """
    증상과 운동 후보를 바탕으로 최종 답변을 생성하는 에이전트
    """

    symptoms = state["symptoms"]
    exercise_candidates = state["exercise_candidates"][:4]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
            너는 운동 추천 답변을 생성하는 에이전트다.

            반드시 아래 형식을 그대로 지켜라.
            출력은 짧고 명확하게 작성한다.

            [사용자 상태 요약]
            - 상태 요약 1
            - 상태 요약 2

            [추천 운동 리스트]
            - 운동명: 추천 이유
            - 운동명: 추천 이유
            - 운동명: 추천 이유
            - 운동명: 추천 이유

            [운동 시작 방법]
            - 시작 방법 1
            - 시작 방법 2

            [주의사항]
            - 주의사항 1
            - 주의사항 2

            작성 조건:
            1. 반드시 입력받은 운동 후보를 모두 포함한다.
            2. 추천 이유는 운동마다 한 문장만 작성한다.
            3. 전체 답변은 12줄 이내로 작성한다.
            4. 마크다운 제목은 #, ## 대신 [] 형식만 사용한다.
            5. 의학적 진단처럼 단정하지 않는다.
            """
        ),
        (
            "human",
            """
            사용자 증상:
            {symptoms}

            반드시 포함할 운동 후보:
            {exercise_candidates}
            """
        )
    ])

    chain = prompt | llm | parser

    final_answer = chain.invoke({
        "symptoms": ", ".join(symptoms),
        "exercise_candidates": ", ".join(exercise_candidates)
    })

    print("\n==============================")
    print("[3] 답변 생성 에이전트")
    print("==============================")
    print(final_answer)

    return {
        "final_answer": final_answer
    }


# 6. LangGraph 그래프 구성

graph_builder = StateGraph(ExerciseState)

# 노드 추가
graph_builder.add_node("증상_추출_에이전트", symptom_extractor_agent)
graph_builder.add_node("운동_후보_에이전트", exercise_candidate_agent)
graph_builder.add_node("답변_생성_에이전트", answer_generator_agent)

# 흐름 연결
graph_builder.add_edge(START, "증상_추출_에이전트")
graph_builder.add_edge("증상_추출_에이전트", "운동_후보_에이전트")
graph_builder.add_edge("운동_후보_에이전트", "답변_생성_에이전트")
graph_builder.add_edge("답변_생성_에이전트", END)

# 그래프 컴파일
app = graph_builder.compile()



display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 8. 사용자 질문 입력 및 실행

user_question = "체력이 안좋고, 살이 계속 찌는데 어떤 운동을 할까?"

initial_state = {
    "user_question": user_question,
    "symptoms": [],
    "exercise_candidates": [],
    "final_answer": ""
}

print("==============================================")
print("LangGraph Multi-Agent Exercise Recommendation")
print("==============================================")
print("[사용자 질문]")
print(user_question)

result = app.invoke(initial_state)

print("\n==============================================")
print("[최종 답변]")
print("==============================================")
print(result["final_answer"])